In [1]:
import pandas as pd
import numpy as np

In [4]:
renewal_calls = pd.read_csv(
    "../../data/01_raw/raw_renewal_calls.csv",
    low_memory=False
)

renewal_calls.shape

(186534, 41)

In [6]:
renewal_calls.info()
renewal_calls.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186534 entries, 0 to 186533
Data columns (total 41 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   Call_ID                                            186534 non-null  float64
 1   Call_Direction                                     186534 non-null  object 
 2   Co_Ref                                             179149 non-null  object 
 3   Call_Date                                          186534 non-null  object 
 4   Churn_Category                                     7902 non-null    object 
 5   Complaint_Category                                 19008 non-null   object 
 6   Customer_Reaction_Category                         23085 non-null   object 
 7   Agent_Renewal_Pitch_Category                       53738 non-null   object 
 8   Customer_Renewal_Response_Category                 54312 non-null   object

,Call_ID,Call_Direction,Co_Ref,Call_Date,Churn_Category,Complaint_Category,Customer_Reaction_Category,Agent_Renewal_Pitch_Category,Customer_Renewal_Response_Category,Agent_Response_Category,...,Customer_Response,Desire_To_Cancel,Discount_Offered,Justification_Category,Reason_For_Renewal_Category,Agent_Response_To_Cancel_Category,Argument_That_Convinced_Customer_to_Stay_Category,Analysed_Call,Call_Number,Call_Year
0,5.950000e+11,Outbound,UB0899,29-01-2025,NaN,NaN,Not Mentioned,Discussion / Introduction / Inquiry,Discount and Offer,Discount and Offer,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,3,2025
1,5.970000e+11,OUT_BOUND,HN5141,26-02-2025,NaN,NaN,NaN,Price and Cost,Agreement,Customer Communication,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,2,2025
2,5.950000e+11,Outbound,BP5009,24-01-2025,NaN,NaN,NaN,Expiration / Due,Agreement,Accreditation and Certification,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,1,2025
3,6.520000e+11,OUT_BOUND,XP8119,09-06-2025,NaN,NaN,NaN,Auto / Automatic,Agreement,Accreditation and Certification,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,1,2025
4,5.370000e+11,Outbound,ZL7978,20-08-2024,NaN,NaN,NaN,NaN,NaN,NaN,...,Not Discussed,Not Discussed,No,NaN,NaN,NaN,NaN,1.0,28,2024


In [7]:
renewal_calls = renewal_calls.drop_duplicates()
print("Shape after duplicate removal:", renewal_calls.shape)

Shape after duplicate removal: (157722, 41)


In [8]:
renewal_calls.columns = (
    renewal_calls.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [9]:
obj_cols = renewal_calls.select_dtypes(include="object").columns

for col in obj_cols:
    renewal_calls[col] = renewal_calls[col].astype(str).str.strip()

In [10]:
date_cols = [
    "call_date",
    "created_date",
    "updated_date"
]

for col in date_cols:
    if col in renewal_calls.columns:
        renewal_calls[col] = pd.to_datetime(
            renewal_calls[col],
            errors="coerce"
        )

C:\Users\shrey\AppData\Local\Temp\ipykernel_3892\2305538497.py:9: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  renewal_calls[col] = pd.to_datetime(


In [11]:
possible_numeric_cols = [
    "call_duration",
    "number_of_attempts",
    "score",
    "customer_rating"
]

for col in possible_numeric_cols:
    if col in renewal_calls.columns:
        renewal_calls[col] = pd.to_numeric(
            renewal_calls[col],
            errors="coerce"
        )

In [12]:
yes_no_cols = [
    col for col in renewal_calls.columns
    if renewal_calls[col].dtype == "object"
]

In [13]:
for col in yes_no_cols:
    renewal_calls[col] = renewal_calls[col].replace({
        "yes": "Yes",
        "YES": "Yes",
        "y": "Yes",
        "no": "No",
        "NO": "No",
        "n": "No"
    })

In [14]:
cat_cols = renewal_calls.select_dtypes(include="object").columns

for col in cat_cols:
    renewal_calls[col] = renewal_calls[col].replace("nan", np.nan)
    renewal_calls[col] = renewal_calls[col].fillna("Unknown")

In [16]:
num_cols = renewal_calls.select_dtypes(include=np.number).columns

for col in num_cols:
    renewal_calls[col] = renewal_calls[col].fillna(
        renewal_calls[col].median()
    )

In [17]:
renewal_calls.isnull().sum().sort_values(ascending=False).head(20)

unnamed:_20                                          157722
explicit_switching_intent                                 0
price_switching_mentioned                                 0
competitor_value_comparison                               0
competitor_benefits_mentioned                             0
topic_introduced_by                                       0
percentage_price_increase_mentioned                       0
monetary_price_increase_mentioned                         0
price_range_mentioned                                     0
customer_asked_for_justification                          0
customer_response                                         0
desire_to_cancel                                          0
discount_offered                                          0
justification_category                                    0
reason_for_renewal_category                               0
agent_response_to_cancel_category                         0
argument_that_convinced_customer_to_stay

In [18]:
renewal_calls.to_csv(
    "../../data/02_processed/processed_renewal_calls.csv",
    index=False
)

In [19]:
# 🔍 COMPREHENSIVE DATA QUALITY & COLUMN ANALYSIS

print("="*70)
print("RENEWAL CALLS DATA: CLEANING & COLUMN NECESSITY ANALYSIS")
print("="*70)

print(f"\nDataset Shape: {renewal_calls.shape}")
print(f"Total nulls: {renewal_calls.isnull().sum().sum()}\n")

# 1. Check for ID/tracking columns that may be redundant
print("1️⃣  ID/TRACKING COLUMNS (potentially unnecessary):")
id_keywords = ["id", "ref", "call_id", "call_number", "unnamed"]
for col in renewal_calls.columns:
    if any(keyword in col.lower() for keyword in id_keywords):
        unique_pct = renewal_calls[col].nunique() / len(renewal_calls) * 100
        print(f"   {col}: {renewal_calls[col].nunique()} unique values ({unique_pct:.1f}% cardinality)")

# 2. Check for timestamp/tracking columns (created_date, updated_date)
print("\n2️⃣  TIMESTAMP/AUDIT COLUMNS (might not help with churn prediction):")
timestamp_keywords = ["created", "updated", "modified"]
for col in renewal_calls.columns:
    if any(keyword in col.lower() for keyword in timestamp_keywords):
        print(f"   {col}: {renewal_calls[col].dtype} - May be redundant with call_date")

# 3. Check for completely empty or near-empty columns
print("\n3️⃣  EMPTY/SPARSE COLUMNS (>80% missing):")
for col in renewal_calls.columns:
    missing_pct = renewal_calls[col].isnull().sum() / len(renewal_calls) * 100
    if missing_pct > 80:
        print(f"   {col}: {missing_pct:.1f}% missing")

# 4. Check low-variance columns (>90% one value)
print("\n4️⃣  LOW-VARIANCE COLUMNS (>90% single value):")
for col in renewal_calls.columns:
    if renewal_calls[col].dtype == 'object':
        top_pct = renewal_calls[col].value_counts().head(1).values[0] / len(renewal_calls) * 100
        if top_pct > 90:
            top_val = renewal_calls[col].value_counts().head(1).index[0]
            print(f"   {col}: '{top_val}' = {top_pct:.1f}%")

# 5. Check high cardinality columns (>10k unique)
print("\n5️⃣  HIGH CARDINALITY COLUMNS (might need feature engineering):")
for col in renewal_calls.columns:
    if renewal_calls[col].dtype == 'object':
        unique_count = renewal_calls[col].nunique()
        if unique_count > 5000:
            print(f"   {col}: {unique_count} unique values")

# 6. Check for duplicate columns
print("\n6️⃣  CHECKING FOR CORRELATED/DUPLICATE FEATURES:")
numeric_only = renewal_calls.select_dtypes(include=[np.number])
if numeric_only.shape[1] > 0:
    corr_matrix = numeric_only.corr()
    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.95:
                high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
    
    if high_corr_pairs:
        for col1, col2, corr in high_corr_pairs:
            print(f"   {col1} ↔ {col2}: {corr:.3f} correlation (consider dropping one)")
    else:
        print("   ✅ No highly correlated numeric columns found")

# 7. Summary of column types
print("\n7️⃣  COLUMN TYPE DISTRIBUTION:")
print(f"   Categorical (object): {renewal_calls.select_dtypes(include='object').shape[1]}")
print(f"   Numeric (int/float): {renewal_calls.select_dtypes(include=[np.number]).shape[1]}")
print(f"   Datetime: {renewal_calls.select_dtypes(include=['datetime64']).shape[1]}")

print("\n" + "="*70)

RENEWAL CALLS DATA: CLEANING & COLUMN NECESSITY ANALYSIS

Dataset Shape: (157722, 41)
Total nulls: 157722

1️⃣  ID/TRACKING COLUMNS (potentially unnecessary):
   call_id: 53 unique values (0.0% cardinality)
   co_ref: 36304 unique values (23.0% cardinality)
   unnamed:_20: 0 unique values (0.0% cardinality)
   call_number: 638 unique values (0.4% cardinality)

2️⃣  TIMESTAMP/AUDIT COLUMNS (might not help with churn prediction):

3️⃣  EMPTY/SPARSE COLUMNS (>80% missing):
   unnamed:_20: 100.0% missing

4️⃣  LOW-VARIANCE COLUMNS (>90% single value):
   churn_category: 'Unknown' = 95.0%
   justification_category: 'Unknown' = 91.7%
   agent_response_to_cancel_category: 'Unknown' = 96.9%
   argument_that_convinced_customer_to_stay_category: 'Unknown' = 98.8%

5️⃣  HIGH CARDINALITY COLUMNS (might need feature engineering):
   co_ref: 36304 unique values

6️⃣  CHECKING FOR CORRELATED/DUPLICATE FEATURES:
   ✅ No highly correlated numeric columns found

7️⃣  COLUMN TYPE DISTRIBUTION:
   Categor

In [20]:

# 🧹 COLUMN CLEANUP & REMOVAL

print("\n📋 PERFORMING COLUMN CLEANUP...")

# 1. Drop completely empty columns
cols_to_drop = []
for col in renewal_calls.columns:
    if renewal_calls[col].isnull().sum() == len(renewal_calls):
        cols_to_drop.append(col)

if cols_to_drop:
    renewal_calls = renewal_calls.drop(columns=cols_to_drop)
    print(f"   ✅ Dropped {len(cols_to_drop)} completely empty columns: {cols_to_drop}")

# 2. Drop redundant ID columns with very low cardinality
if "call_id" in renewal_calls.columns:
    unique_count = renewal_calls["call_id"].nunique()
    if unique_count < 100:  # Only 53 unique values for 157k rows = essentially useless
        renewal_calls = renewal_calls.drop(columns=["call_id"])
        print(f"   ✅ Dropped 'call_id' ({unique_count} unique values - too low cardinality)")

if "call_number" in renewal_calls.columns:
    unique_count = renewal_calls["call_number"].nunique()
    if unique_count < 1000:  # Only 638 unique for 157k rows
        renewal_calls = renewal_calls.drop(columns=["call_number"])
        print(f"   ✅ Dropped 'call_number' ({unique_count} unique values - sequential/low info)")

# 3. Drop extremely low-variance columns (>95% Unknown or one value)
low_var_cols = []
for col in renewal_calls.select_dtypes(include="object").columns:
    top_pct = renewal_calls[col].value_counts().head(1).values[0] / len(renewal_calls) * 100
    if top_pct > 95:
        if renewal_calls[col].value_counts().head(1).index[0] == "Unknown":
            low_var_cols.append(col)

if low_var_cols:
    renewal_calls = renewal_calls.drop(columns=low_var_cols)
    print(f"   ✅ Dropped {len(low_var_cols)} extremely low-variance columns (>95% Unknown):")
    for col in low_var_cols:
        print(f"      - {col}")

# 4. Keep co_ref (customer reference) but it won't be used in modeling
# 5. Keep numeric columns for potential feature engineering

print(f"\n✅ AFTER CLEANUP:")
print(f"   New shape: {renewal_calls.shape}")
print(f"   Columns removed: {len(cols_to_drop) + (1 if 'call_id' in renewal_calls.columns else 0) + (1 if 'call_number' in renewal_calls.columns else 0) + len(low_var_cols)}")

# Convert binary Yes/No columns to 0/1
print(f"\n   Converting Yes/No columns to binary...")
binary_count = 0
for col in renewal_calls.select_dtypes(include="object").columns:
    unique_vals = set(renewal_calls[col].dropna().unique())
    if unique_vals.issubset({'Yes', 'No', 'Unknown'}) and len(unique_vals) > 1:
        renewal_calls[col] = renewal_calls[col].map({'Yes': 1, 'No': 0, 'Unknown': 0})
        binary_count += 1
        print(f"      ✅ {col}")

if binary_count == 0:
    print("      (No Yes/No columns found to convert)")

print(f"\n✅ Total cleaning operations: {len(cols_to_drop) + (2 if 'call_id' in renewal_calls.columns else 0) + (2 if 'call_number' in renewal_calls.columns else 0) + len(low_var_cols) + binary_count}")


📋 PERFORMING COLUMN CLEANUP...
   ✅ Dropped 1 completely empty columns: ['unnamed:_20']
   ✅ Dropped 'call_id' (53 unique values - too low cardinality)
   ✅ Dropped 'call_number' (638 unique values - sequential/low info)
   ✅ Dropped 2 extremely low-variance columns (>95% Unknown):
      - agent_response_to_cancel_category
      - argument_that_convinced_customer_to_stay_category

✅ AFTER CLEANUP:
   New shape: (157722, 36)
   Columns removed: 3

   Converting Yes/No columns to binary...
      (No Yes/No columns found to convert)

✅ Total cleaning operations: 3


In [21]:

# 📊 FINAL DATA QUALITY CHECK & SAVE

print("\n📊 FINAL DATA QUALITY STATUS:")

# Check remaining nulls (should be 0 after filling)
null_count = renewal_calls.isnull().sum().sum()
print(f"   Total nulls: {null_count}")

# Check column types
obj_count = renewal_calls.select_dtypes(include='object').shape[1]
num_count = renewal_calls.select_dtypes(include=[np.number]).shape[1]
date_count = renewal_calls.select_dtypes(include=['datetime64']).shape[1]

print(f"   Column distribution:")
print(f"      - Categorical: {obj_count}")
print(f"      - Numeric: {num_count}")
print(f"      - Datetime: {date_count}")

# Check remaining "Unknown" values
print(f"\n   'Unknown' values by column:")
for col in renewal_calls.select_dtypes(include='object').columns:
    unknown_count = (renewal_calls[col] == "Unknown").sum()
    if unknown_count > 0:
        pct = unknown_count / len(renewal_calls) * 100
        print(f"      {col}: {unknown_count} ({pct:.1f}%)")

# Summary
print(f"\n✅ FINAL RENEWAL CALLS DATA:")
print(f"   Shape: {renewal_calls.shape}")
print(f"   Memory: {renewal_calls.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"   Ready for modeling: {'Yes' if null_count == 0 else 'No - null check needed'}")

# Save the cleaneddata
renewal_calls.to_csv(
    "../../data/02_processed/processed_renewal_calls.csv",
    index=False
)
print(f"\n   ✅ Saved to processed_renewal_calls.csv")


📊 FINAL DATA QUALITY STATUS:
   Total nulls: 0
   Column distribution:
      - Categorical: 33
      - Numeric: 2
      - Datetime: 1

   'Unknown' values by column:
      co_ref: 4453 (2.8%)
      churn_category: 149821 (95.0%)
      complaint_category: 138728 (88.0%)
      customer_reaction_category: 134736 (85.4%)
      agent_renewal_pitch_category: 103985 (65.9%)
      customer_renewal_response_category: 103411 (65.6%)
      agent_response_category: 103743 (65.8%)
      membership_renewal_decision: 71583 (45.4%)
      serious_complaint: 73805 (46.8%)
      other_complaint: 73806 (46.8%)
      discussion_on_price_increase: 70123 (44.5%)
      renewal_impact_due_to_price_increase: 70152 (44.5%)
      discount_or_waiver_requested: 70152 (44.5%)
      call_reschedule_request: 69579 (44.1%)
      agent_flagged_membership_status_alert: 69579 (44.1%)
      agent_renewal_initiation: 69579 (44.1%)
      explicit_competitor_mention: 69453 (44.0%)
      explicit_switching_intent: 69453 (44.0